# Deepfake Forensics Analysis (Goal 7)

Uses CRISE saliency maps to diagnose *why* synthetic probes succeed or fail at fooling ArcFace.

**Pipeline:**
1. Match synthetic probe saliency maps to real probe saliency maps (same identity)
2. Compute saliency divergence: cosine similarity + L1
3. Establish similarity threshold empirically from the distribution
4. Classify every synthetic probe into case A / B / C / D
5. Per-region importance analysis (5 facial zones on 112×112 aligned chips)
6. Cross-method comparison table + 8+ figures

**Case taxonomy:**

| Case | Fooled ArcFace? | Saliency similar? | Interpretation |
|---|---|---|---|
| A | Yes | Yes | Fooled for right reasons — genuine identity features replicated |
| B | Yes | No  | **Fooled for wrong reasons** — exploiting artifacts / non-identity regions |
| C | No  | Yes | Correct features but insufficient identity transfer |
| D | No  | No  | Complete failure |

**Inputs required (must exist before running):**
- `data/synthetic_probes/metadata.csv`
- `results/crise_maps/synth_saliency_index.csv`
- `results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv`
- `results/crise_maps/*_saliency_norm.npy`

In [ ]:
# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

META_PATH        = "data/synthetic_probes/metadata.csv"
SAL_INDEX_PATH   = "results/crise_maps/synth_saliency_index.csv"
REAL_SUMMARY     = "results/crise_maps/summary_crise_tau0.1_N1000_s8_p0.5_MASTERSEED123.csv"
CRISE_MAP_DIR    = "results/crise_maps"

FIG_DIR          = "results/forensics_figures"

# Primary alpha/strength for per-method comparisons
PRIMARY_ALPHA    = 0.5

# SIM_THRESHOLD is derived empirically in the threshold-derivation cell below.
# Do NOT set it here — it is overwritten by the 5th-pct of real intra-identity
# CRISE cosine similarity.  Value at last run: 0.853
SIM_THRESHOLD    = None   # placeholder; overwritten below

In [ ]:
# ---------------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------------

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from tqdm import tqdm
from scipy import stats

os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
# ---------------------------------------------------------------------------
# Facial region masks (fixed in 112×112 aligned chip space)
# Based on _DST_LANDMARKS positions from rise.py:
#   left eye (38,52), right eye (74,52), nose (56,72),
#   mouth-L (42,92), mouth-R (71,92)
# ---------------------------------------------------------------------------

def make_region_masks(h=112, w=112):
    """Return dict of (h,w) float32 binary region masks for the aligned chip."""
    masks = {}
    y, x = np.mgrid[0:h, 0:w]

    def ellipse(cy, cx, ry, rx):
        return (((y - cy) / ry) ** 2 + ((x - cx) / rx) ** 2 <= 1).astype(np.float32)

    # Eye zone: covers both eyes + brow area
    masks["eyes"]     = np.clip(ellipse(50, 38, 18, 20) + ellipse(50, 74, 18, 20), 0, 1)
    # Nose zone
    masks["nose"]     = ellipse(72, 56, 16, 14).astype(np.float32)
    # Mouth zone
    masks["mouth"]    = ellipse(92, 56, 14, 22).astype(np.float32)
    # Forehead / upper face (above eyes)
    masks["forehead"] = ((y < 38) & (x > 15) & (x < 97)).astype(np.float32)
    # Jaw / chin (below mouth)
    masks["jaw_chin"] = ((y > 98) & (x > 15) & (x < 97)).astype(np.float32)

    return masks


REGION_MASKS  = make_region_masks()
REGION_NAMES  = ["forehead", "eyes", "nose", "mouth", "jaw_chin"]

def region_importance(sal_norm):
    """Return dict of fractional saliency weight per region."""
    total = sal_norm.sum() + 1e-12
    return {r: float((sal_norm * REGION_MASKS[r]).sum() / total) for r in REGION_NAMES}


# Quick sanity: visualise masks
fig, axes = plt.subplots(1, len(REGION_NAMES), figsize=(12, 2.5))
for ax, name in zip(axes, REGION_NAMES):
    ax.imshow(REGION_MASKS[name], cmap="viridis", vmin=0, vmax=1)
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.suptitle("Region masks (112×112 aligned chip)", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig_region_masks.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

meta      = pd.read_csv(META_PATH)
sal_index = pd.read_csv(SAL_INDEX_PATH)
real_sum  = pd.read_csv(REAL_SUMMARY)

# Normalise column name for real summary
if "true_id" not in real_sum.columns:
    real_sum = real_sum.rename(columns={real_sum.columns[0]: "true_id"})

print(f"Synthetic probes      : {len(meta)}")
print(f"Saliency index entries: {len(sal_index)}")
print(f"Real probe CRISE rows : {len(real_sum)}")

In [ ]:
# ---------------------------------------------------------------------------
# Build per-identity real probe saliency map (mean over all real probes)
# ---------------------------------------------------------------------------

# real_sum must have a 'saliency_path' or we reconstruct from the file naming convention
# Try 'saliency_path' column first; fall back to scanning crise_maps dir

real_sal_by_id = {}   # identity -> mean saliency map (112×112 float32)

if "saliency_path" in real_sum.columns:
    sal_col = "saliency_path"
else:
    # Column may be named differently — find any column ending in .npy path
    npy_cols = [c for c in real_sum.columns if real_sum[c].astype(str).str.endswith(".npy").any()]
    sal_col = npy_cols[0] if npy_cols else None

if sal_col:
    for _, row in tqdm(real_sum.iterrows(), total=len(real_sum), desc="Loading real saliency maps"):
        tid  = row["true_id"]
        path = row[sal_col]
        if not isinstance(path, str) or not os.path.exists(path):
            continue
        sal = np.load(path).astype(np.float32)
        if tid not in real_sal_by_id:
            real_sal_by_id[tid] = []
        real_sal_by_id[tid].append(sal)

    # Mean over all real probes per identity
    real_sal_by_id = {tid: np.mean(maps, axis=0)
                      for tid, maps in real_sal_by_id.items()}
    print(f"Loaded real saliency maps for {len(real_sal_by_id)} identities")
else:
    print("[WARN] No saliency_path column found in real summary — skipping real map load.")
    print("       Update REAL_SUMMARY to point to a CSV that includes saliency_path.")

In [ ]:
# ---------------------------------------------------------------------------
# Principled saliency similarity threshold
#
# A binary "saliency similar / divergent" cutoff must be empirically grounded,
# not arbitrary.  We derive it from real intra-identity CRISE map variation:
#   for every identity with 2+ real probe maps, compute pairwise cosine
#   similarity between those maps.  The threshold is set at the 5th percentile
#   of this distribution.
#
# Interpretation: a synthetic probe's saliency map is "divergent" (Cases B/D)
# only if it falls below the range of variation seen in 95% of genuine same-
# identity real-probe pairs.  This is the smallest defensible threshold —
# anything looser would flag normal cross-probe variation as divergent.
#
# Empirical values (251 identities, 3213 pairs):
#   mean = 0.913   std = 0.032   5th pct = 0.853
# ---------------------------------------------------------------------------

from itertools import combinations
import re

_crise_real_files = [
    f for f in os.listdir(CRISE_MAP_DIR)
    if f.endswith('_saliency_norm.npy') and f.startswith('crise_tau0p1_')
]

# Parse identity — real probes have numeric image IDs like _0001_N; synthetics don't
_id_to_maps = {}
for f in _crise_real_files:
    stem = f.replace('crise_tau0p1_', '').replace('_saliency_norm.npy', '')
    m = re.search(r'^(.+?)_(\w+_\d{4})_N', stem)
    if m:
        _id_to_maps.setdefault(m.group(1), []).append(os.path.join(CRISE_MAP_DIR, f))

_multi = {k: v for k, v in _id_to_maps.items() if len(v) >= 2}
print(f"Identities with 2+ real CRISE maps: {len(_multi)}")

intra_sims = []
for _maps in _multi.values():
    loaded = [np.load(p).ravel().astype(np.float32) for p in _maps]
    for a, b in combinations(loaded, 2):
        an = a / (np.linalg.norm(a) + 1e-9)
        bn = b / (np.linalg.norm(b) + 1e-9)
        intra_sims.append(float(an @ bn))

intra_sims = np.array(intra_sims)
_p5  = float(np.percentile(intra_sims, 5))
_p25 = float(np.percentile(intra_sims, 25))
_med = float(np.median(intra_sims))

print(f"Real intra-identity pairs : {len(intra_sims)}")
print(f"  mean={intra_sims.mean():.3f}  std={intra_sims.std():.3f}")
print(f"  5th pct={_p5:.3f}   25th pct={_p25:.3f}   median={_med:.3f}")

# Overwrite the placeholder
SIM_THRESHOLD = round(_p5, 3)
print(f"
SIM_THRESHOLD set to {SIM_THRESHOLD}  (5th percentile of real intra-identity sim)")

# Cache distribution for the reference figure
np.save(os.path.join(FIG_DIR, "intra_identity_crise_sim.npy"), intra_sims)

# ── Reference figure: real intra-identity sim distribution with threshold line ──
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(intra_sims, bins=60, color='#4E79A7', alpha=0.8, density=True)
ax.axvline(SIM_THRESHOLD, color='#E15759', lw=2, ls='--',
           label=f'5th pct threshold = {SIM_THRESHOLD:.3f}')
ax.axvline(_med, color='#F28E2B', lw=1.5, ls=':',
           label=f'median = {_med:.3f}')
ax.set_xlabel("Cosine similarity (real probe pairs, same identity)", fontsize=10)
ax.set_ylabel("Density", fontsize=10)
ax.set_title("Intra-identity CRISE saliency similarity
"
             "(calibration distribution for SIM_THRESHOLD)", fontsize=11)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
_thresh_fig = os.path.join(FIG_DIR, "fig0_threshold_calibration.png")
plt.savefig(_thresh_fig, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {_thresh_fig}")

In [ ]:
# ---------------------------------------------------------------------------
# Compute saliency divergence for each synthetic probe
# ---------------------------------------------------------------------------

rows_out = []

for _, srow in tqdm(sal_index.iterrows(), total=len(sal_index), desc="Saliency divergence"):
    tid       = srow["identity"]
    sal_path  = srow["saliency_path"]

    if not isinstance(sal_path, str) or not os.path.exists(sal_path):
        continue

    synth_sal = np.load(sal_path).astype(np.float32).ravel()
    synth_sal_2d = synth_sal.reshape(112, 112)

    real_sal_2d  = real_sal_by_id.get(tid)
    if real_sal_2d is None:
        continue

    real_flat = real_sal_2d.ravel()

    # Cosine similarity (both already in [0,1], normalise for dot product)
    s_norm = synth_sal / (np.linalg.norm(synth_sal) + 1e-12)
    r_norm = real_flat / (np.linalg.norm(real_flat) + 1e-12)
    cos_sim = float(np.dot(s_norm, r_norm))
    l1_dist = float(np.abs(synth_sal - real_flat).mean())

    # Per-region importance
    reg_imp = region_importance(synth_sal_2d)

    rows_out.append(dict(
        output_path      = srow["output_path"],
        identity         = tid,
        generation_method= srow["generation_method"],
        blend_alpha      = srow["blend_alpha"],
        saliency_path    = sal_path,
        saliency_cosine_sim = cos_sim,
        saliency_l1         = l1_dist,
        **{f"reg_{k}": v for k, v in reg_imp.items()},
    ))

div_df = pd.DataFrame(rows_out)
print(f"Divergence computed for {len(div_df)} synthetic probes")

In [ ]:
# ---------------------------------------------------------------------------
# Merge divergence back into metadata
# ---------------------------------------------------------------------------

# Drop any stale columns from meta before merging
stale_cols = [c for c in meta.columns
              if c.startswith("reg_") or c in ("saliency_cosine_sim", "saliency_l1", "saliency_path")]
meta_clean = meta.drop(columns=stale_cols)

merge_cols = (["output_path", "saliency_path", "saliency_cosine_sim", "saliency_l1"] +
              [f"reg_{r}" for r in REGION_NAMES])
meta_upd = meta_clean.merge(div_df[merge_cols], on="output_path", how="left")

meta_upd.to_csv(META_PATH, index=False)
print(f"metadata.csv updated → {META_PATH}")
print("Columns added:", [c for c in meta_upd.columns if c.startswith("reg_") or "saliency" in c])

In [ ]:
# ---------------------------------------------------------------------------
# Fig 1: Saliency cosine similarity distribution
# Used to set SIM_THRESHOLD empirically
# ---------------------------------------------------------------------------

sims = div_df["saliency_cosine_sim"].dropna()

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(sims, bins=40, color="steelblue", edgecolor="white", linewidth=0.4)
ax.axvline(sims.median(), color="red",    ls="--", lw=1.5, label=f"Median = {sims.median():.3f}")
ax.axvline(sims.quantile(0.25), color="orange", ls=":", lw=1.2,
           label=f"25th pct = {sims.quantile(0.25):.3f}")
ax.set_xlabel("Cosine similarity (synthetic vs real probe saliency maps)")
ax.set_ylabel("Count")
ax.set_title("Saliency map similarity distribution — all synthetic probes")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig1_saliency_sim_dist.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"mean={sims.mean():.3f}  median={sims.median():.3f}  std={sims.std():.3f}")
print(f"Suggested threshold (median): {sims.median():.3f}")
print(">>> Update SIM_THRESHOLD in Config cell if needed, then re-run from the case classification cell.")

In [ ]:
# ---------------------------------------------------------------------------
# Case classification  (update SIM_THRESHOLD above if needed)
# ---------------------------------------------------------------------------

analysis = meta_upd[
    meta_upd["embedding_ok"].astype(bool) &
    meta_upd["rank1_match"].notna() &
    meta_upd["saliency_cosine_sim"].notna()
].copy()

analysis["sal_similar"] = analysis["saliency_cosine_sim"] >= SIM_THRESHOLD

def classify_case(row):
    if row["rank1_match"] and row["sal_similar"]:     return "A"
    if row["rank1_match"] and not row["sal_similar"]: return "B"
    if not row["rank1_match"] and row["sal_similar"]: return "C"
    return "D"

analysis["case_label"] = analysis.apply(classify_case, axis=1)

# Write case labels back to metadata
meta_upd.loc[analysis.index, "case_label"] = analysis["case_label"]
meta_upd.to_csv(META_PATH, index=False)

print(f"Classified {len(analysis)} probes  (threshold={SIM_THRESHOLD})")
print(analysis["case_label"].value_counts().sort_index().to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Fig 2: Cross-method comparison table (Case A/B/C/D % + Rank-1 rate)
# Primary alpha/strength only
# ---------------------------------------------------------------------------

primary = analysis[
    (analysis["blend_alpha"] == PRIMARY_ALPHA) |
    (analysis["generation_method"] == "insightface_swap")  # no alpha for swap
].copy()

rows_table = []
for method, grp in primary.groupby("generation_method"):
    total = len(grp)
    row = {"Method": method, "n": total,
           "Rank-1 rate": grp["rank1_match"].mean()}
    for case in ["A", "B", "C", "D"]:
        row[f"Case {case} %"] = (grp["case_label"] == case).sum() / total * 100
    rows_table.append(row)

table_df = pd.DataFrame(rows_table).set_index("Method")
print("=== Cross-method comparison ===")
print(table_df.round(3).to_string())

# Bar chart
case_cols = ["Case A %", "Case B %", "Case C %", "Case D %"]
colors    = ["#2ecc71", "#e74c3c", "#f39c12", "#95a5a6"]

fig, ax = plt.subplots(figsize=(8, 4))
table_df[case_cols].plot(kind="bar", ax=ax, color=colors, edgecolor="white", width=0.7)
ax.set_ylabel("% of probes")
ax.set_title(f"Case distribution per generation method (α/strength = {PRIMARY_ALPHA})")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
ax.legend(loc="upper right", fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig2_cross_method_cases.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Fig 3: Per-region importance profiles by case
# ---------------------------------------------------------------------------

reg_cols = [f"reg_{r}" for r in REGION_NAMES]
case_reg = analysis.groupby("case_label")[reg_cols].mean()

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(REGION_NAMES))
width = 0.18
case_colors = {"A": "#2ecc71", "B": "#e74c3c", "C": "#f39c12", "D": "#95a5a6"}

for i, (case, row_) in enumerate(case_reg.iterrows()):
    ax.bar(x + i * width, row_.values, width, label=f"Case {case}",
           color=case_colors.get(case, "gray"), edgecolor="white")

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(REGION_NAMES)
ax.set_ylabel("Mean fractional saliency weight")
ax.set_title("Per-region importance by forensic case")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig3_region_importance_by_case.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nMean per-region importance by case:")
print(case_reg.round(4).to_string())

In [ ]:
# ---------------------------------------------------------------------------
# Fig 4: Saliency similarity vs ArcFace similarity (scatter, coloured by case)
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6, 5))
for case, grp in analysis.groupby("case_label"):
    ax.scatter(grp["arcface_similarity"], grp["saliency_cosine_sim"],
               label=f"Case {case}", alpha=0.5, s=18,
               color=case_colors.get(case, "gray"))

ax.axhline(SIM_THRESHOLD, color="black", ls="--", lw=1, label=f"Threshold={SIM_THRESHOLD:.2f}")
ax.set_xlabel("ArcFace cosine similarity to true identity")
ax.set_ylabel("Saliency map cosine similarity (synthetic vs real)")
ax.set_title("ArcFace confidence vs saliency faithfulness")
ax.legend(fontsize=8, markerscale=1.5)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "fig4_sim_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Fig 5 & 6: Alpha/strength ablation — rank-1 rate and case distribution
# ---------------------------------------------------------------------------

for method in analysis["generation_method"].unique():
    grp_m = analysis[analysis["generation_method"] == method]
    if grp_m["blend_alpha"].nunique() < 2:
        continue

    ablation_rows = []
    for a, grp_a in grp_m.groupby("blend_alpha"):
        r = {"alpha": a, "rank1_rate": grp_a["rank1_match"].mean()}
        for case in ["A", "B", "C", "D"]:
            r[f"case_{case}"] = (grp_a["case_label"] == case).mean() * 100
        ablation_rows.append(r)
    abl_df = pd.DataFrame(ablation_rows).sort_values("alpha")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))

    ax1.plot(abl_df["alpha"], abl_df["rank1_rate"], "o-", color="steelblue")
    ax1.set_xlabel("alpha / strength")
    ax1.set_ylabel("Rank-1 rate")
    ax1.set_title(f"{method} — fooling rate vs alpha")
    ax1.grid(alpha=0.3)

    for case, color in case_colors.items():
        ax2.plot(abl_df["alpha"], abl_df[f"case_{case}"], "o-",
                 color=color, label=f"Case {case}")
    ax2.set_xlabel("alpha / strength")
    ax2.set_ylabel("% of probes")
    ax2.set_title(f"{method} — case distribution vs alpha")
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)

    plt.suptitle(f"Ablation: {method}", fontsize=11)
    plt.tight_layout()
    fname = f"fig_ablation_{method}.png"
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Figs 7+: Example saliency map pairs — 2 examples per case (8 figures)
# Shows: real probe saliency | synthetic saliency | overlay difference
# ---------------------------------------------------------------------------

N_EXAMPLES = 2

for case in ["A", "B", "C", "D"]:
    case_rows = analysis[analysis["case_label"] == case].dropna(
        subset=["saliency_path"]
    ).head(N_EXAMPLES)

    if len(case_rows) == 0:
        print(f"No examples for Case {case}")
        continue

    fig, axes = plt.subplots(len(case_rows), 3,
                             figsize=(9, 3 * len(case_rows)))
    if len(case_rows) == 1:
        axes = axes[np.newaxis, :]

    for row_i, (_, row_) in enumerate(case_rows.iterrows()):
        tid = row_["identity"]

        synth_sal = np.load(row_["saliency_path"]).astype(np.float32)
        real_sal  = real_sal_by_id.get(tid)

        diff = np.abs(synth_sal - real_sal) if real_sal is not None else np.zeros_like(synth_sal)

        titles = [
            f"Real probe\n{tid[:22]}",
            f"Synthetic ({row_['generation_method']})\ncos_sim={row_['saliency_cosine_sim']:.3f}",
            f"|Δ saliency|\nrank1={row_['rank1_match']}  case={case}",
        ]
        imgs = [real_sal if real_sal is not None else np.zeros_like(synth_sal),
                synth_sal, diff]

        for col_i, (img, title) in enumerate(zip(imgs, titles)):
            axes[row_i, col_i].imshow(img, cmap="hot", vmin=0, vmax=1)
            axes[row_i, col_i].set_title(title, fontsize=8)
            axes[row_i, col_i].axis("off")

    plt.suptitle(f"Case {case} examples", fontsize=11)
    plt.tight_layout()
    fname = f"fig_examples_case{case}.png"
    plt.savefig(os.path.join(FIG_DIR, fname), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Case {case}: saved {fname}")

In [ ]:
# ---------------------------------------------------------------------------
# Summary table (paper-ready)
# ---------------------------------------------------------------------------

print("=== Full cross-method summary (all alphas/strengths) ===")
full_table = []
for method, grp in analysis.groupby("generation_method"):
    total = len(grp)
    r = {"Method": method, "n": total,
         "Rank-1 rate": f"{grp['rank1_match'].mean():.3f}"}
    for case in ["A", "B", "C", "D"]:
        pct = (grp["case_label"] == case).sum() / total * 100
        r[f"Case {case} %"] = f"{pct:.1f}"
    full_table.append(r)

print(pd.DataFrame(full_table).set_index("Method").to_string())

print("\n=== Primary alpha only ===")
print(table_df.round(3).to_string())

print(f"\nAll figures saved to: {FIG_DIR}/")

---
## Fig 8 — Case A Deep Dive: Real vs. Synthetic Individual CRISE Comparison

The existing saliency divergence metric compares each synthetic map to the **mean** real-probe map per identity. This cell uses **individual probe pairs** — directly matching each Case A synthetic to the specific real probe it is paired with. This is what a court would actually want: was ArcFace looking at the same facial evidence for both the real photo and the deepfake?

Four examples are shown: the 2 most similar and 2 most divergent Case A pairs by cosine similarity.

In [ ]:
# ---------------------------------------------------------------------------
# Fig 8 — Case A deep dive: individual real vs. synthetic CRISE comparison
# ---------------------------------------------------------------------------

case_a_rows = analysis[analysis["case_label"] == "A"].copy()
print(f"Case A probes: {len(case_a_rows)}")

# Build identity -> list of (probe_path, sal_path) from real_sum
real_probe_paths = {}
for _, row in real_sum.iterrows():
    sp = row.get("saliency_path", None)
    if isinstance(sp, str) and os.path.exists(sp):
        real_probe_paths.setdefault(row["true_id"], []).append(
            (row["probe_path"], sp)
        )

pair_records = []
for _, row in case_a_rows.iterrows():
    identity  = row["identity"]
    synth_sal = row.get("saliency_path") if isinstance(row.get("saliency_path"), str) else None
    if synth_sal is None or not os.path.exists(synth_sal):
        continue
    real_options = real_probe_paths.get(identity, [])
    if not real_options:
        continue

    synth_map = np.load(synth_sal).astype(np.float32)
    sv = synth_map.ravel(); sv /= (np.linalg.norm(sv) + 1e-9)

    real_ppath, real_sal_path = real_options[0]
    real_map = np.load(real_sal_path).astype(np.float32)
    rv = real_map.ravel(); rv /= (np.linalg.norm(rv) + 1e-9)

    indiv_cos = float(rv @ sv)

    reg_div = {}
    for rname, mask in REGION_MASKS.items():
        r_frac = float(real_map[mask].sum() / (real_map.sum() + 1e-9))
        s_frac = float(synth_map[mask].sum() / (synth_map.sum() + 1e-9))
        reg_div[rname] = abs(r_frac - s_frac)

    pair_records.append({
        "identity":        identity,
        "synth_path":      row["output_path"],
        "real_probe_path": real_ppath,
        "real_sal_path":   real_sal_path,
        "synth_sal_path":  synth_sal,
        "indiv_cos":       indiv_cos,
        "method":          row["generation_method"],
        "blend_alpha":     row["blend_alpha"],
        "arcface_sim":     row["arcface_similarity"],
        **{f"reg_div_{k}": v for k, v in reg_div.items()},
    })

pair_df = pd.DataFrame(pair_records).sort_values("indiv_cos", ascending=False)
print(f"Pairs evaluated: {len(pair_df)}")
print(f"  cos sim — mean={pair_df['indiv_cos'].mean():.3f}  "
      f"std={pair_df['indiv_cos'].std():.3f}  min={pair_df['indiv_cos'].min():.3f}")

reg_div_cols = [c for c in pair_df.columns if c.startswith("reg_div_")]
print("\nMean per-region |divergence| across Case A individual pairs:")
for c in reg_div_cols:
    print(f"  {c.replace('reg_div_', ''):12s}  {pair_df[c].mean():.4f}")

pair_df.to_csv(os.path.join(FIG_DIR, "case_a_individual_pairs.csv"), index=False)

# ── Visualise 2 best + 2 worst Case A pairs ─────────────────────────────
top2    = pair_df.head(2)
bottom2 = pair_df.tail(2)
showcase = pd.concat([top2, bottom2], ignore_index=True)
row_labels = ["Best (1)", "Best (2)", "Worst (1)", "Worst (2)"]

from insightface.app import FaceAnalysis as _FA
_app_fig8 = _FA(name="buffalo_l",
                providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
_app_fig8.prepare(ctx_id=0, det_size=(640, 640))

fig, axes = plt.subplots(4, 4, figsize=(13, 13))
col_titles = ["Real chip", "Real CRISE", "Synth chip", "Synth CRISE"]
for j, t in enumerate(col_titles):
    axes[0, j].set_title(t, fontsize=10, fontweight="bold")

for i, (_, pr) in enumerate(showcase.iterrows()):
    real_img  = cv2.imread(pr["real_probe_path"])
    synth_img = cv2.imread(pr["synth_path"])
    real_map  = np.load(pr["real_sal_path"]).astype(np.float32)
    synth_map = np.load(pr["synth_sal_path"]).astype(np.float32)

    def _chip(img):
        from rise import build_aligned_chip_112
        if img is None:
            return np.zeros((112, 112, 3), dtype=np.uint8)
        try:
            return build_aligned_chip_112(img, _app_fig8)
        except Exception:
            return np.zeros((112, 112, 3), dtype=np.uint8)

    rc = _chip(real_img);  sc = _chip(synth_img)
    axes[i, 0].imshow(cv2.cvtColor(rc, cv2.COLOR_BGR2RGB))
    axes[i, 1].imshow(real_map,  cmap="inferno", vmin=0, vmax=1)
    axes[i, 2].imshow(cv2.cvtColor(sc, cv2.COLOR_BGR2RGB))
    axes[i, 3].imshow(synth_map, cmap="inferno", vmin=0, vmax=1)

    title_color = "#2ecc71" if i < 2 else "#e74c3c"
    axes[i, 0].set_ylabel(
        f"{row_labels[i]}\n{pr['identity']}\n({pr['method']})",
        fontsize=7, rotation=0, labelpad=90, va="center")
    axes[i, 1].set_title(f"arcface={pr['arcface_sim']:.3f}", fontsize=8)
    axes[i, 3].set_title(f"cos={pr['indiv_cos']:.3f}", fontsize=9,
                          color=title_color, fontweight="bold")

for ax in axes.ravel():
    ax.axis("off")

plt.suptitle(
    "Fig 8 — Case A: Real vs. Synthetic CRISE maps\n"
    "Top 2: most similar pairs  |  Bottom 2: most divergent pairs",
    fontsize=12, y=1.01)
plt.tight_layout()
_fig8 = os.path.join(FIG_DIR, "fig8_case_a_individual_pairs.png")
plt.savefig(_fig8, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig8}")


---
## Fig 9 — Identity Vulnerability Analysis

Which identities are structurally more at risk from morphing and SD img2img attacks?  
Each target identity has 3 synthetic probes per method. We classify identities as  
**vulnerable** (all 3 fooled), **partial** (1–2 fooled), or **resistant** (0 fooled).

This is the societal-impact finding made concrete: CRISE-ID identifies *which individuals* face greater forensic risk from deepfake attacks. Connect to `demographic_analysis.ipynb` to test whether vulnerability correlates with estimated gender or age bracket.

In [ ]:
# ---------------------------------------------------------------------------
# Fig 9 — Identity vulnerability analysis
# ---------------------------------------------------------------------------

vuln_rows = []
for identity, grp in meta.groupby("identity"):
    for method, mgrp in grp.groupby("generation_method"):
        if method == "insightface_swap":
            primary = mgrp
        else:
            primary = mgrp[mgrp["blend_alpha"] == 0.5]
        if primary.empty:
            continue
        r1 = int(primary["rank1_match"].sum())
        n  = len(primary)
        if   r1 == n:  status = "Vulnerable"
        elif r1 == 0:  status = "Resistant"
        else:          status = "Partial"
        vuln_rows.append({"identity": identity, "method": method,
                          "n_probes": n, "n_fooled": r1,
                          "status": status, "fool_rate": r1 / n})

vuln_df = pd.DataFrame(vuln_rows)
print("Vulnerability distribution (α=0.5 / all for swap):")
print(vuln_df.groupby(["method", "status"]).size().unstack(fill_value=0))
vuln_df.to_csv(os.path.join(FIG_DIR, "identity_vulnerability.csv"), index=False)

# ── Figure ────────────────────────────────────────────────────────────────
methods_plot  = [m for m in ["morphing", "sd_img2img", "insightface_swap"]
                 if m in vuln_df["method"].unique()]
status_colors = {"Vulnerable": "#e74c3c", "Partial": "#f39c12", "Resistant": "#2ecc71"}
status_order  = ["Vulnerable", "Partial", "Resistant"]

fig, axes = plt.subplots(1, len(methods_plot),
                          figsize=(4.5 * len(methods_plot), 4.5))
if len(methods_plot) == 1:
    axes = [axes]

for ax, method in zip(axes, methods_plot):
    sub   = vuln_df[vuln_df["method"] == method]
    total = len(sub)
    bars  = [sub["status"].value_counts().get(s, 0) for s in status_order]
    pcts  = [100 * b / total for b in bars]
    ax.bar(status_order, pcts,
           color=[status_colors[s] for s in status_order],
           alpha=0.85, edgecolor="white", linewidth=0.8)
    for s, p, b in zip(status_order, pcts, bars):
        ax.text(s, p + 0.5, str(b), ha="center", va="bottom", fontsize=10)
    ax.set_title(method.replace("_", " ").title(), fontsize=11)
    ax.set_ylabel("% of target identities", fontsize=9)
    ax.set_ylim(0, max(pcts) * 1.2 + 5)
    ax.grid(axis="y", alpha=0.3)

plt.suptitle(
    "Fig 9 — Identity vulnerability to synthetic attacks  (primary α / strength = 0.5)",
    fontsize=12)
plt.tight_layout()
_fig9 = os.path.join(FIG_DIR, "fig9_identity_vulnerability.png")
plt.savefig(_fig9, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig9}")

morph_vuln = vuln_df[vuln_df["method"] == "morphing"].sort_values("fool_rate", ascending=False)
print("\nTop 10 most vulnerable identities (morphing α=0.5):")
print(morph_vuln.head(10)[["identity", "n_fooled", "n_probes", "status"]].to_string(index=False))
print("\n10 most resistant identities:")
print(morph_vuln.tail(10)[["identity", "n_fooled", "n_probes", "status"]].to_string(index=False))


---
## Fig 10 — Identity Absorption Curve

The central novel visual: saliency cosine similarity (line) and rank-1 success rate (bars) as a function of generation strength, for both attack types.

- **Morphing (left):** monotonic — more target identity blended → higher saliency sim → higher rank-1
- **SD img2img (right):** *inverted* — low strength barely alters the original (trivially fools ArcFace); high strength generates a new AI face that ArcFace rejects. Same model, opposite forensic outcome.

In [ ]:
# ---------------------------------------------------------------------------
# Fig 10 — Identity absorption curve
# Dual-axis per method: rank-1 rate (bars) + mean saliency cosine sim (line ± 1 std)
# ---------------------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
method_colors = {"morphing": "#4E79A7", "sd_img2img": "#E15759"}

for ax, method in zip(axes, ["morphing", "sd_img2img"]):
    sub    = meta[(meta["generation_method"] == method) &
                  meta["saliency_cosine_sim"].notna()].copy()
    alphas = sorted(sub["blend_alpha"].dropna().unique())
    r1_pct = [sub[sub["blend_alpha"] == a]["rank1_match"].mean() * 100 for a in alphas]
    cs_avg = [sub[sub["blend_alpha"] == a]["saliency_cosine_sim"].mean() for a in alphas]
    cs_std = [sub[sub["blend_alpha"] == a]["saliency_cosine_sim"].std()  for a in alphas]

    color = method_colors[method]
    x = np.arange(len(alphas))
    ax2 = ax.twinx()

    bars = ax.bar(x, r1_pct, width=0.4, alpha=0.55, color=color, label="Rank-1 rate (%)")
    line, = ax2.plot(x, cs_avg, "o-", color="black", lw=2.5, label="Saliency cos sim", zorder=5)
    ax2.fill_between(x, [c - s for c, s in zip(cs_avg, cs_std)],
                         [c + s for c, s in zip(cs_avg, cs_std)],
                     color="black", alpha=0.1)

    ax.set_xticks(x)
    ax.set_xticklabels([str(a) for a in alphas], fontsize=12)
    xlabel = "Blend ratio α (morphing)" if method == "morphing" else "Generation strength"
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel("Rank-1 success rate (%)", fontsize=10, color=color)
    ax.tick_params(axis="y", labelcolor=color)
    ax.set_ylim(0, 115)
    ax2.set_ylabel("Mean saliency cosine similarity", fontsize=10)
    ax2.set_ylim(0.80, 0.98)

    title = ("Morphing — identity absorption\n"
             "(higher α → more identity transferred)")
    if method == "sd_img2img":
        title = ("SD img2img — behavioral inversion\n"
                 "(higher strength → new AI face → lower rank-1)")
    ax.set_title(title, fontsize=10)
    ax.grid(axis="y", alpha=0.25)

    legend_loc = "upper left" if method == "morphing" else "upper right"
    ax.legend([bars, line], ["Rank-1 rate (%)", "Mean saliency cos sim ± 1 std"],
              loc=legend_loc, fontsize=8, framealpha=0.9)

plt.suptitle("Fig 10 — Identity absorption curve: saliency similarity tracks identity transfer",
             fontsize=12, y=1.02)
plt.tight_layout()
_fig10 = os.path.join(FIG_DIR, "fig10_identity_absorption_curve.png")
plt.savefig(_fig10, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {_fig10}")

for method in ["morphing", "sd_img2img"]:
    sub = meta[(meta["generation_method"] == method) & meta["saliency_cosine_sim"].notna()]
    for a in sorted(sub["blend_alpha"].dropna().unique()):
        g = sub[sub["blend_alpha"] == a]
        print(f"  [{method}] strength/alpha={a}  "
              f"rank1={g['rank1_match'].mean()*100:.0f}%  "
              f"sal_cos={g['saliency_cosine_sim'].mean():.3f}±{g['saliency_cosine_sim'].std():.3f}")


---
## Fig 11 — RISE vs. CRISE: Identity Tracking on Synthetic Probes

CRISE outperforms RISE on real probes (insertion/deletion AUC). This section extends that to the forensic setting: for Case A synthetic probes (deepfakes that fool ArcFace), is the real probe's **CRISE** map more similar to the synthetic's CRISE map than the real probe's **RISE** map is?

Metric: `cos(real_CRISE, synth_CRISE)` vs `cos(real_RISE, synth_CRISE)`.  
If CRISE-CRISE > RISE-CRISE, CRISE better captures the identity evidence the deepfake is exploiting — a forensic advantage unavailable to baseline RISE.

Statistical test: Wilcoxon signed-rank (paired, non-parametric).

In [ ]:
# ---------------------------------------------------------------------------
# Fig 11 — RISE vs. CRISE: which method better tracks synthetic identity transfer?
# ---------------------------------------------------------------------------

RISE_SUMMARY_PATH = "results/rise_baseline/summary_K1680_N1000_s8_p0.5_MASTERSEED123.csv"
rise_sum_df = pd.read_csv(RISE_SUMMARY_PATH)
rise_sum_df = rise_sum_df[
    rise_sum_df["saliency_path"].notna() &
    rise_sum_df["saliency_path"].apply(os.path.exists)
].copy()

rise_real_sal = {row["true_id"]: row["saliency_path"]
                 for _, row in rise_sum_df.iterrows()}

from scipy.stats import wilcoxon

records_rc = []
for _, row in analysis[analysis["case_label"] == "A"].iterrows():
    identity  = row["identity"]
    synth_sal = row.get("saliency_path") if isinstance(row.get("saliency_path"), str) else None
    if synth_sal is None or not os.path.exists(synth_sal):
        continue
    real_crise_opts = real_probe_paths.get(identity, [])
    if not real_crise_opts:
        continue
    real_rise_path = rise_real_sal.get(identity)
    if real_rise_path is None or not os.path.exists(real_rise_path):
        continue

    try:
        def _nload(p):
            v = np.load(p).ravel().astype(np.float32)
            return v / (np.linalg.norm(v) + 1e-9)

        sv  = _nload(synth_sal)
        cv  = _nload(real_crise_opts[0][1])
        rv  = _nload(real_rise_path)

        records_rc.append({
            "identity":        identity,
            "method":          row["generation_method"],
            "blend_alpha":     row["blend_alpha"],
            "cos_crise_synth": float(cv @ sv),
            "cos_rise_synth":  float(rv @ sv),
            "cos_crise_rise":  float(cv @ rv),
            "crise_advantage": float(cv @ sv) - float(rv @ sv),
            "arcface_sim":     row["arcface_similarity"],
        })
    except Exception:
        pass

rc_df = pd.DataFrame(records_rc)
print(f"Case A pairs with both RISE + CRISE real maps: {len(rc_df)}")

if len(rc_df) > 0:
    m_c = rc_df["cos_crise_synth"].mean()
    m_r = rc_df["cos_rise_synth"].mean()
    adv = rc_df["crise_advantage"].mean()
    stat, pval = wilcoxon(rc_df["cos_crise_synth"], rc_df["cos_rise_synth"])
    print(f"  mean cos(real_CRISE, synth) = {m_c:.4f}")
    print(f"  mean cos(real_RISE,  synth) = {m_r:.4f}")
    print(f"  CRISE advantage             = {adv:+.4f}")
    print(f"  Wilcoxon: stat={stat:.1f}, p={pval:.4f}")
    rc_df.to_csv(os.path.join(FIG_DIR, "rise_vs_crise_on_synthetics.csv"), index=False)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    # Panel 1: paired scatter
    ax = axes[0]
    sc = ax.scatter(rc_df["cos_rise_synth"], rc_df["cos_crise_synth"],
                    alpha=0.5, s=20, c=rc_df["arcface_sim"], cmap="viridis")
    plt.colorbar(sc, ax=ax, label="ArcFace similarity")
    lim_lo = min(rc_df[["cos_rise_synth", "cos_crise_synth"]].min()) - 0.01
    lim_hi = max(rc_df[["cos_rise_synth", "cos_crise_synth"]].max()) + 0.01
    ax.plot([lim_lo, lim_hi], [lim_lo, lim_hi], "k--", lw=1, label="Equal")
    frac_above = (rc_df["crise_advantage"] > 0).mean()
    ax.set_xlabel("cos(real RISE, synth CRISE)", fontsize=10)
    ax.set_ylabel("cos(real CRISE, synth CRISE)", fontsize=10)
    ax.set_title(f"Points above diagonal: CRISE tracks better\n({frac_above*100:.0f}% of pairs)", fontsize=9)
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    # Panel 2: advantage histogram
    ax2 = axes[1]
    ax2.hist(rc_df["crise_advantage"], bins=30, color="#4E79A7", alpha=0.8, edgecolor="white")
    ax2.axvline(0,   color="k",       lw=1.5, ls="--", label="No difference")
    ax2.axvline(adv, color="#E15759", lw=2,
                label=f"mean={adv:+.4f}  (p={pval:.4f})")
    ax2.set_xlabel("CRISE advantage  (cos_CRISE − cos_RISE)", fontsize=10)
    ax2.set_ylabel("Count", fontsize=10)
    ax2.set_title("CRISE advantage distribution across Case A pairs", fontsize=10)
    ax2.legend(fontsize=9); ax2.grid(alpha=0.3)

    plt.suptitle("Fig 11 — RISE vs. CRISE: identity tracking on Case A synthetic probes",
                 fontsize=12)
    plt.tight_layout()
    _fig11 = os.path.join(FIG_DIR, "fig11_rise_vs_crise_synthetics.png")
    plt.savefig(_fig11, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {_fig11}")
else:
    print("No matched pairs — check real_probe_paths and rise_real_sal are populated.")
